# Create a Personal Chef assistant which can help user to suggest a recipe. Also it can answer any followup questions. 

In [ ]:
import os
import uuid
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain_core.tools import tool
from pprint import pprint
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.checkpoint.memory import InMemorySaver
load_dotenv(override=True)

We will use local LLM rather than OpenAI API. For search we will use Tavily LLM. Langsmith will be used for observality.Hence make sure below environment variables are defined in .env file. 
* TAVILY_API_KEY
* LANGSMITH_PROJECT
* LANGSMITH_API_KEY
* LOCAL_LLM_URL

In [ ]:
# Load the environment varaibles  #noqa
llm = ChatOpenAI(
    model="qwen2.5:32b",
    base_url=os.getenv("LOCAL_LLM_URL"),
    api_key="NA",
)

In [ ]:
# Lets define tool for search
from tavily import TavilyClient
tavily_client = TavilyClient()

@tool("search", description="Call this tool to when user search for food recipe")
def food_recipe(query: str):
    return tavily_client.search(query)

In [ ]:
# create agent using local llm, tools with memory
config = {"configurable": {"thread_id": uuid.uuid4()}}
agent = create_agent(model=llm, tools=[food_recipe], checkpointer=InMemorySaver())

In [ ]:
user_text = input("What all ingrediant left with you: ")

In [ ]:
system_message = SystemMessage("You are a head chef. User will share the ingredients list and you need to propose a receipy along with steps.Then user may have subsequent queries")
user_message = HumanMessage(f"Ingrediants available: {user_text}")

In [ ]:
response = agent.invoke({"messages":[system_message, user_message]}, config=config)

In [ ]:
pprint(response.get("messages")[-1])

In [ ]:
user_query = input("do you have any other question: ")
user_query = HumanMessage(f"Any query: {user_query}")

In [ ]:
response = agent.invoke({"messages":[user_query]}, config)

In [ ]:
pprint(response["messages"][-1])